In [ ]:
import csv
import os
from datetime import datetime
from gradio.flagging import FlaggingCallback

In [ ]:
class BadCaseLogger(FlaggingCallback):
    def setup(self, components, flagging_dir):
        self.flagging_dir = flagging_dir or "flagged_recipes"
        os.makedirs(self.flagging_dir, exist_ok=True)
        self.csv_path = os.path.join(self.flagging_dir, "bad_recipes.csv")
        
        # 헤더 없으면 생성
        if not os.path.exists(self.csv_path):
            with open(self.csv_path, "w", newline="", encoding="utf-8") as f:
                writer = csv.writer(f)
                writer.writerow(["timestamp", "ingredients", "model_output", "flag_option", "username"])

    def flag(self, flag_data, flag_option=None, username=None):
        # flag_data 예: [ingredients_text, recipe_text]
        ingredients, recipe = flag_data
        
        with open(self.csv_path, "a", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow([
                datetime.now().isoformat(),
                ingredients,
                recipe,
                flag_option,
                username
            ])
        
        print("[FLAG] 저장됨:", ingredients[:30], "...")

In [ ]:
demo = gr.Interface(
    fn=generate_recipe,                   # 너의 Gemini 레시피 함수
    inputs="text",
    outputs="text",
    allow_flagging="manual",
    flagging_options=["맛없음", "너무어려움", "재료이상함"],
    flagging_callback=BadCaseLogger()
)
